# Taller Algoritmos Geneticos

Juan David Ramirez

Juan Diego Carreño

## Importacion de Librerias

In [ ]:
!pip install deap

In [ ]:
import numpy as np
import random
import time
import matplotlib.pyplot as plt
from IPython import display
from deap import base, creator, tools, algorithms

## Definición del laberinto y funciones base

In [ ]:
# Laberinto 20x20
laberinto = np.zeros((20, 20))
inicio = (0, 0)
salida = (0, 19)

for i in range(1, 19, 4):
    laberinto[0:19, i] = 1
for i in range(3, 20, 4):
    laberinto[1:20, i] = 1

# Movimientos posibles: arriba, abajo, izquierda, derecha, nada
MOVIMIENTOS = [(0, 1), (0, -1), (-1, 0), (1, 0), (0, 0)]


def mostrar_laberinto():
    ax = plt.gca()
    ax.imshow(laberinto.T, cmap='binary')
    plt.gca().invert_yaxis()
    plt.plot(inicio[0], inicio[1], "ro", label="Inicio")
    plt.plot(salida[0], salida[1], "go", label="Salida")
    plt.grid(True, which='both')
    plt.legend()
    plt.show()


def mover(pos, movimiento):
    nueva_pos = (pos[0] + movimiento[0], pos[1] + movimiento[1])
    if 0 <= nueva_pos[0] < laberinto.shape[0] and 0 <= nueva_pos[1] < laberinto.shape[1]:
        if laberinto[nueva_pos] == 0:
            return nueva_pos
    return pos


def evaluar_individuo(individual):
    pos = inicio
    for movimiento in individual:
        pos = mover(pos, MOVIMIENTOS[movimiento])
    dist_salida = np.sqrt((pos[0] - salida[0])**2 + (pos[1] - salida[1])**2)
    return dist_salida,


def mostrar_ruta_pasos(individuo, s=0.01, laberinto=None, inicio=None, salida=None, grid=False):
    generacion = individuo if isinstance(individuo[0], list) else [individuo]
    plt.clf()
    fig, ax = plt.subplots()

    for ind in generacion:
        pos = inicio
        camino = [pos]
        for i, movimiento in enumerate(ind):
            pos = mover(pos, MOVIMIENTOS[movimiento])
            camino.append(pos)
            camino_np = np.array(camino)

            ax.clear()
            if grid:
                ax.set_xticks(np.arange(0, laberinto.shape[1], 1))
                ax.set_yticks(np.arange(0, laberinto.shape[1], 1))
                ax.grid(grid, which='both')
            ax.set_ylim(-1, laberinto.shape[1])
            ax.set_xlim(-1, laberinto.shape[0])
            ax.imshow(laberinto.T, cmap='binary')
            ax.plot(camino_np[:, 0], camino_np[:, 1], "r-")
            ax.plot(-1, -1, "ro", label=f"{i+1}")
            ax.plot(inicio[0], inicio[1], "bo", label="Inicio")
            ax.plot(salida[0], salida[1], "go", label="Salida")
            ax.plot(pos[0], pos[1], "bx", label=f"pos {pos}")
            plt.xlabel("Eje X")
            plt.ylabel("Eje Y")
            ax.legend()

            display.clear_output(wait=True)
            display.display(fig)
            time.sleep(s)
    plt.close(fig)


# Mostrar laberinto para verificar
mostrar_laberinto()

## Configuración del algoritmo genético

In [ ]:
# Limpiar clases previas si existen (evita errores al reejecutar)
if hasattr(creator, "FitnessMin"):
    del creator.FitnessMin
if hasattr(creator, "Individual"):
    del creator.Individual

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
toolbox.register("attr_move", random.randint, 0, len(MOVIMIENTOS) - 1)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutShuffleIndexes, indpb=0.1)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", evaluar_individuo)

## Parámetros editables

### Experimentos: Tamaño de la población

#### Población pequeña = 20

In [ ]:
# ============================================
#   PARÁMETROS DEL EXPERIMENTO (EDITABLES)
# ============================================
POP_SIZE = 20      # Tamaño de la población
NGEN     = 300       # Número de generaciones
CXPB     = 0.5      # Probabilidad de cruce
MUTPB    = 0.5      # Probabilidad de mutación
IND_SIZE = 1000       # Tamaño del individuo (cantidad de movimientos)
# ============================================

# Registrar individuo y población con el IND_SIZE actual
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_move, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
# Estadísticas
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("min", np.min)
stats.register("avg", np.mean)

# Población inicial
random.seed()  # Cambiar a un entero para reproducibilidad
pop = toolbox.population(n=POP_SIZE)

# Ejecutar algoritmo genético
resultados, logbook = algorithms.eaSimple(
    pop, toolbox,
    cxpb=CXPB,
    mutpb=MUTPB,
    ngen=NGEN,
    stats=stats,
    verbose=False
)

# Mejor individuo
mejor_individuo = tools.selBest(pop, 1)[0]
mejor_fitness = mejor_individuo.fitness.values[0]

# Posición final del mejor individuo
pos_final = inicio
for mov in mejor_individuo:
    pos_final = mover(pos_final, MOVIMIENTOS[mov])

# Reporte
print("=" * 50)
print("PARÁMETROS DEL EXPERIMENTO")
print("=" * 50)
print(f"POP_SIZE : {POP_SIZE}")
print(f"NGEN     : {NGEN}")
print(f"CXPB     : {CXPB}")
print(f"MUTPB    : {MUTPB}")
print(f"IND_SIZE : {IND_SIZE}")
print("=" * 50)
print("RESULTADOS")
print("=" * 50)
print(f"Mejor aptitud (distancia)  : {mejor_fitness:.4f}")
print(f"Posición final alcanzada   : {pos_final}")
print(f"Posición de salida (meta)  : {salida}")
print(f"Distancia final a la meta  : {mejor_fitness:.4f}")
print("=" * 50)

In [ ]:
 # --- Gráfica de evolución ---
generaciones = logbook.select("gen")
minimos      = logbook.select("min")
promedios    = logbook.select("avg")

plt.figure(figsize=(8, 5))
plt.plot(generaciones, minimos, label="Mejor aptitud")
plt.plot(generaciones, promedios, label="Aptitud promedio", linestyle="--")
plt.xlabel("Generaciones")
plt.ylabel("Aptitud (distancia a la salida)")
plt.title(f"Evolución | POP={POP_SIZE}, NGEN={NGEN}, CXPB={CXPB}, MUTPB={MUTPB}, IND={IND_SIZE}")
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización del mejor camino ---
mostrar_ruta_pasos(mejor_individuo, laberinto=laberinto,
                   inicio=inicio, salida=salida, grid=True)

#### Población media = 300

In [ ]:
# ============================================
#   PARÁMETROS DEL EXPERIMENTO (EDITABLES)
# ============================================
POP_SIZE = 300      # Tamaño de la población
NGEN     = 300       # Número de generaciones
CXPB     = 0.5      # Probabilidad de cruce
MUTPB    = 0.5      # Probabilidad de mutación
IND_SIZE = 1000       # Tamaño del individuo (cantidad de movimientos)
# ============================================

# Registrar individuo y población con el IND_SIZE actual
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_move, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
# Estadísticas
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("min", np.min)
stats.register("avg", np.mean)

# Población inicial
random.seed()  # Cambiar a un entero para reproducibilidad
pop = toolbox.population(n=POP_SIZE)

# Ejecutar algoritmo genético
resultados, logbook = algorithms.eaSimple(
    pop, toolbox,
    cxpb=CXPB,
    mutpb=MUTPB,
    ngen=NGEN,
    stats=stats,
    verbose=False
)

# Mejor individuo
mejor_individuo = tools.selBest(pop, 1)[0]
mejor_fitness = mejor_individuo.fitness.values[0]

# Posición final del mejor individuo
pos_final = inicio
for mov in mejor_individuo:
    pos_final = mover(pos_final, MOVIMIENTOS[mov])

# Reporte
print("=" * 50)
print("PARÁMETROS DEL EXPERIMENTO")
print("=" * 50)
print(f"POP_SIZE : {POP_SIZE}")
print(f"NGEN     : {NGEN}")
print(f"CXPB     : {CXPB}")
print(f"MUTPB    : {MUTPB}")
print(f"IND_SIZE : {IND_SIZE}")
print("=" * 50)
print("RESULTADOS")
print("=" * 50)
print(f"Mejor aptitud (distancia)  : {mejor_fitness:.4f}")
print(f"Posición final alcanzada   : {pos_final}")
print(f"Posición de salida (meta)  : {salida}")
print(f"Distancia final a la meta  : {mejor_fitness:.4f}")
print("=" * 50)

In [ ]:
 # --- Gráfica de evolución ---
generaciones = logbook.select("gen")
minimos      = logbook.select("min")
promedios    = logbook.select("avg")

plt.figure(figsize=(8, 5))
plt.plot(generaciones, minimos, label="Mejor aptitud")
plt.plot(generaciones, promedios, label="Aptitud promedio", linestyle="--")
plt.xlabel("Generaciones")
plt.ylabel("Aptitud (distancia a la salida)")
plt.title(f"Evolución | POP={POP_SIZE}, NGEN={NGEN}, CXPB={CXPB}, MUTPB={MUTPB}, IND={IND_SIZE}")
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización del mejor camino ---
mostrar_ruta_pasos(mejor_individuo, laberinto=laberinto,
                   inicio=inicio, salida=salida, grid=True)

#### Población grande = 800

In [ ]:
# ============================================
#   PARÁMETROS DEL EXPERIMENTO (EDITABLES)
# ============================================
POP_SIZE = 800      # Tamaño de la población
NGEN     = 300       # Número de generaciones
CXPB     = 0.5      # Probabilidad de cruce
MUTPB    = 0.5      # Probabilidad de mutación
IND_SIZE = 1000       # Tamaño del individuo (cantidad de movimientos)
# ============================================

# Registrar individuo y población con el IND_SIZE actual
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_move, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
# Estadísticas
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("min", np.min)
stats.register("avg", np.mean)

# Población inicial
random.seed()  # Cambiar a un entero para reproducibilidad
pop = toolbox.population(n=POP_SIZE)

# Ejecutar algoritmo genético
resultados, logbook = algorithms.eaSimple(
    pop, toolbox,
    cxpb=CXPB,
    mutpb=MUTPB,
    ngen=NGEN,
    stats=stats,
    verbose=False
)

# Mejor individuo
mejor_individuo = tools.selBest(pop, 1)[0]
mejor_fitness = mejor_individuo.fitness.values[0]

# Posición final del mejor individuo
pos_final = inicio
for mov in mejor_individuo:
    pos_final = mover(pos_final, MOVIMIENTOS[mov])

# Reporte
print("=" * 50)
print("PARÁMETROS DEL EXPERIMENTO")
print("=" * 50)
print(f"POP_SIZE : {POP_SIZE}")
print(f"NGEN     : {NGEN}")
print(f"CXPB     : {CXPB}")
print(f"MUTPB    : {MUTPB}")
print(f"IND_SIZE : {IND_SIZE}")
print("=" * 50)
print("RESULTADOS")
print("=" * 50)
print(f"Mejor aptitud (distancia)  : {mejor_fitness:.4f}")
print(f"Posición final alcanzada   : {pos_final}")
print(f"Posición de salida (meta)  : {salida}")
print(f"Distancia final a la meta  : {mejor_fitness:.4f}")
print("=" * 50)

In [ ]:
 # --- Gráfica de evolución ---
generaciones = logbook.select("gen")
minimos      = logbook.select("min")
promedios    = logbook.select("avg")

plt.figure(figsize=(8, 5))
plt.plot(generaciones, minimos, label="Mejor aptitud")
plt.plot(generaciones, promedios, label="Aptitud promedio", linestyle="--")
plt.xlabel("Generaciones")
plt.ylabel("Aptitud (distancia a la salida)")
plt.title(f"Evolución | POP={POP_SIZE}, NGEN={NGEN}, CXPB={CXPB}, MUTPB={MUTPB}, IND={IND_SIZE}")
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización del mejor camino ---
mostrar_ruta_pasos(mejor_individuo, laberinto=laberinto,
                   inicio=inicio, salida=salida, grid=True)

#### Análisis: Tamaño de la población

Lo más llamativo de estos experimentos es que pasar de 20 a 300 individuos no cambió nada: ambas configuraciones terminaron en la misma posición frente al mismo muro, con la misma distancia de 11. La mayor diversidad genética que en teoría debería traer una población más grande no fue suficiente para que algún individuo encontrara la vuelta al obstáculo.

El salto a 800 sí marcó una diferencia, logrando avanzar hasta (0, 12) y reducir la distancia a 7. Pero es una solución cara: el tiempo de cómputo creció considerablemente y la mejora fue marginal. Lo que en realidad pasó fue que con tantos individuos, la probabilidad de que alguno tropiece accidentalmente con el camino correcto aumenta, no porque el algoritmo sea más inteligente, sino porque hay más intentos en paralelo.

En resumen, más población ayuda un poco, pero no resuelve el problema de fondo: la función de aptitud penaliza moverse en la dirección incorrecta aunque hacerlo sea necesario para rodear una pared.

### Experimentos: Número de generaciones

#### Generaciones pequeñas = 50

In [ ]:
# ============================================
#   PARÁMETROS DEL EXPERIMENTO (EDITABLES)
# ============================================
POP_SIZE = 300      # Tamaño de la población
NGEN     = 50       # Número de generaciones
CXPB     = 0.5      # Probabilidad de cruce
MUTPB    = 0.5      # Probabilidad de mutación
IND_SIZE = 1000       # Tamaño del individuo (cantidad de movimientos)
# ============================================

# Registrar individuo y población con el IND_SIZE actual
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_move, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
# Estadísticas
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("min", np.min)
stats.register("avg", np.mean)

# Población inicial
random.seed()  # Cambiar a un entero para reproducibilidad
pop = toolbox.population(n=POP_SIZE)

# Ejecutar algoritmo genético
resultados, logbook = algorithms.eaSimple(
    pop, toolbox,
    cxpb=CXPB,
    mutpb=MUTPB,
    ngen=NGEN,
    stats=stats,
    verbose=False
)

# Mejor individuo
mejor_individuo = tools.selBest(pop, 1)[0]
mejor_fitness = mejor_individuo.fitness.values[0]

# Posición final del mejor individuo
pos_final = inicio
for mov in mejor_individuo:
    pos_final = mover(pos_final, MOVIMIENTOS[mov])

# Reporte
print("=" * 50)
print("PARÁMETROS DEL EXPERIMENTO")
print("=" * 50)
print(f"POP_SIZE : {POP_SIZE}")
print(f"NGEN     : {NGEN}")
print(f"CXPB     : {CXPB}")
print(f"MUTPB    : {MUTPB}")
print(f"IND_SIZE : {IND_SIZE}")
print("=" * 50)
print("RESULTADOS")
print("=" * 50)
print(f"Mejor aptitud (distancia)  : {mejor_fitness:.4f}")
print(f"Posición final alcanzada   : {pos_final}")
print(f"Posición de salida (meta)  : {salida}")
print(f"Distancia final a la meta  : {mejor_fitness:.4f}")
print("=" * 50)

In [ ]:
 # --- Gráfica de evolución ---
generaciones = logbook.select("gen")
minimos      = logbook.select("min")
promedios    = logbook.select("avg")

plt.figure(figsize=(8, 5))
plt.plot(generaciones, minimos, label="Mejor aptitud")
plt.plot(generaciones, promedios, label="Aptitud promedio", linestyle="--")
plt.xlabel("Generaciones")
plt.ylabel("Aptitud (distancia a la salida)")
plt.title(f"Evolución | POP={POP_SIZE}, NGEN={NGEN}, CXPB={CXPB}, MUTPB={MUTPB}, IND={IND_SIZE}")
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización del mejor camino ---
mostrar_ruta_pasos(mejor_individuo, laberinto=laberinto,
                   inicio=inicio, salida=salida, grid=True)

#### Generaciones medias = 500

In [ ]:
# ============================================
#   PARÁMETROS DEL EXPERIMENTO (EDITABLES)
# ============================================
POP_SIZE = 300      # Tamaño de la población
NGEN     = 500       # Número de generaciones
CXPB     = 0.5      # Probabilidad de cruce
MUTPB    = 0.5      # Probabilidad de mutación
IND_SIZE = 1000       # Tamaño del individuo (cantidad de movimientos)
# ============================================

# Registrar individuo y población con el IND_SIZE actual
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_move, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
# Estadísticas
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("min", np.min)
stats.register("avg", np.mean)

# Población inicial
random.seed()  # Cambiar a un entero para reproducibilidad
pop = toolbox.population(n=POP_SIZE)

# Ejecutar algoritmo genético
resultados, logbook = algorithms.eaSimple(
    pop, toolbox,
    cxpb=CXPB,
    mutpb=MUTPB,
    ngen=NGEN,
    stats=stats,
    verbose=False
)

# Mejor individuo
mejor_individuo = tools.selBest(pop, 1)[0]
mejor_fitness = mejor_individuo.fitness.values[0]

# Posición final del mejor individuo
pos_final = inicio
for mov in mejor_individuo:
    pos_final = mover(pos_final, MOVIMIENTOS[mov])

# Reporte
print("=" * 50)
print("PARÁMETROS DEL EXPERIMENTO")
print("=" * 50)
print(f"POP_SIZE : {POP_SIZE}")
print(f"NGEN     : {NGEN}")
print(f"CXPB     : {CXPB}")
print(f"MUTPB    : {MUTPB}")
print(f"IND_SIZE : {IND_SIZE}")
print("=" * 50)
print("RESULTADOS")
print("=" * 50)
print(f"Mejor aptitud (distancia)  : {mejor_fitness:.4f}")
print(f"Posición final alcanzada   : {pos_final}")
print(f"Posición de salida (meta)  : {salida}")
print(f"Distancia final a la meta  : {mejor_fitness:.4f}")
print("=" * 50)

In [ ]:
 # --- Gráfica de evolución ---
generaciones = logbook.select("gen")
minimos      = logbook.select("min")
promedios    = logbook.select("avg")

plt.figure(figsize=(8, 5))
plt.plot(generaciones, minimos, label="Mejor aptitud")
plt.plot(generaciones, promedios, label="Aptitud promedio", linestyle="--")
plt.xlabel("Generaciones")
plt.ylabel("Aptitud (distancia a la salida)")
plt.title(f"Evolución | POP={POP_SIZE}, NGEN={NGEN}, CXPB={CXPB}, MUTPB={MUTPB}, IND={IND_SIZE}")
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización del mejor camino ---
mostrar_ruta_pasos(mejor_individuo, laberinto=laberinto,
                   inicio=inicio, salida=salida, grid=True)

#### Generaciones grandes = 1200

In [ ]:
# ============================================
#   PARÁMETROS DEL EXPERIMENTO (EDITABLES)
# ============================================
POP_SIZE = 300      # Tamaño de la población
NGEN     = 1200       # Número de generaciones
CXPB     = 0.5      # Probabilidad de cruce
MUTPB    = 0.5      # Probabilidad de mutación
IND_SIZE = 1000       # Tamaño del individuo (cantidad de movimientos)
# ============================================

# Registrar individuo y población con el IND_SIZE actual
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_move, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
# Estadísticas
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("min", np.min)
stats.register("avg", np.mean)

# Población inicial
random.seed()  # Cambiar a un entero para reproducibilidad
pop = toolbox.population(n=POP_SIZE)

# Ejecutar algoritmo genético
resultados, logbook = algorithms.eaSimple(
    pop, toolbox,
    cxpb=CXPB,
    mutpb=MUTPB,
    ngen=NGEN,
    stats=stats,
    verbose=False
)

# Mejor individuo
mejor_individuo = tools.selBest(pop, 1)[0]
mejor_fitness = mejor_individuo.fitness.values[0]

# Posición final del mejor individuo
pos_final = inicio
for mov in mejor_individuo:
    pos_final = mover(pos_final, MOVIMIENTOS[mov])

# Reporte
print("=" * 50)
print("PARÁMETROS DEL EXPERIMENTO")
print("=" * 50)
print(f"POP_SIZE : {POP_SIZE}")
print(f"NGEN     : {NGEN}")
print(f"CXPB     : {CXPB}")
print(f"MUTPB    : {MUTPB}")
print(f"IND_SIZE : {IND_SIZE}")
print("=" * 50)
print("RESULTADOS")
print("=" * 50)
print(f"Mejor aptitud (distancia)  : {mejor_fitness:.4f}")
print(f"Posición final alcanzada   : {pos_final}")
print(f"Posición de salida (meta)  : {salida}")
print(f"Distancia final a la meta  : {mejor_fitness:.4f}")
print("=" * 50)

In [ ]:
 # --- Gráfica de evolución ---
generaciones = logbook.select("gen")
minimos      = logbook.select("min")
promedios    = logbook.select("avg")

plt.figure(figsize=(8, 5))
plt.plot(generaciones, minimos, label="Mejor aptitud")
plt.plot(generaciones, promedios, label="Aptitud promedio", linestyle="--")
plt.xlabel("Generaciones")
plt.ylabel("Aptitud (distancia a la salida)")
plt.title(f"Evolución | POP={POP_SIZE}, NGEN={NGEN}, CXPB={CXPB}, MUTPB={MUTPB}, IND={IND_SIZE}")
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización del mejor camino ---
mostrar_ruta_pasos(mejor_individuo, laberinto=laberinto,
                   inicio=inicio, salida=salida, grid=True)

#### Análisis: Número de generaciones

De 50 a 500 generaciones sí hay una mejora visible: el algoritmo necesita tiempo para que los operadores de mutación y cruce puedan ir refinando rutas que tengan alguna oportunidad de bordear el primer muro. Con apenas 50 iteraciones la población termina prácticamente donde empezó (distancia 11), mientras que con 500 logra avanzar hasta (0, 12).

Sin embargo, subir de 500 a 1200 generaciones no produjo ningún cambio adicional: la población quedó fija en exactamente el mismo punto. Esto indica que el algoritmo ya convergió antes de las 500 generaciones y el resto del tiempo es esfuerzo en vano. Una vez que todos los individuos comparten secuencias genéticas muy similares (convergencia prematura), añadir más generaciones no reintroduce diversidad, así que el resultado no mejora.

El patrón de las curvas de evolución también lo confirma: la distancia mínima baja rápido en las primeras generaciones y luego se aplana completamente. Correr más generaciones después de ese punto plano es computacionalmente inútil.

### Experimentos: Tasa de cruce

#### Tasa de cruce baja = 0.1

In [ ]:
# ============================================
#   PARÁMETROS DEL EXPERIMENTO (EDITABLES)
# ============================================
POP_SIZE = 300      # Tamaño de la población
NGEN     = 300       # Número de generaciones
CXPB     = 0.1      # Probabilidad de cruce
MUTPB    = 0.5      # Probabilidad de mutación
IND_SIZE = 1000       # Tamaño del individuo (cantidad de movimientos)
# ============================================

# Registrar individuo y población con el IND_SIZE actual
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_move, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
# Estadísticas
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("min", np.min)
stats.register("avg", np.mean)

# Población inicial
random.seed()  # Cambiar a un entero para reproducibilidad
pop = toolbox.population(n=POP_SIZE)

# Ejecutar algoritmo genético
resultados, logbook = algorithms.eaSimple(
    pop, toolbox,
    cxpb=CXPB,
    mutpb=MUTPB,
    ngen=NGEN,
    stats=stats,
    verbose=False
)

# Mejor individuo
mejor_individuo = tools.selBest(pop, 1)[0]
mejor_fitness = mejor_individuo.fitness.values[0]

# Posición final del mejor individuo
pos_final = inicio
for mov in mejor_individuo:
    pos_final = mover(pos_final, MOVIMIENTOS[mov])

# Reporte
print("=" * 50)
print("PARÁMETROS DEL EXPERIMENTO")
print("=" * 50)
print(f"POP_SIZE : {POP_SIZE}")
print(f"NGEN     : {NGEN}")
print(f"CXPB     : {CXPB}")
print(f"MUTPB    : {MUTPB}")
print(f"IND_SIZE : {IND_SIZE}")
print("=" * 50)
print("RESULTADOS")
print("=" * 50)
print(f"Mejor aptitud (distancia)  : {mejor_fitness:.4f}")
print(f"Posición final alcanzada   : {pos_final}")
print(f"Posición de salida (meta)  : {salida}")
print(f"Distancia final a la meta  : {mejor_fitness:.4f}")
print("=" * 50)

In [ ]:
 # --- Gráfica de evolución ---
generaciones = logbook.select("gen")
minimos      = logbook.select("min")
promedios    = logbook.select("avg")

plt.figure(figsize=(8, 5))
plt.plot(generaciones, minimos, label="Mejor aptitud")
plt.plot(generaciones, promedios, label="Aptitud promedio", linestyle="--")
plt.xlabel("Generaciones")
plt.ylabel("Aptitud (distancia a la salida)")
plt.title(f"Evolución | POP={POP_SIZE}, NGEN={NGEN}, CXPB={CXPB}, MUTPB={MUTPB}, IND={IND_SIZE}")
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización del mejor camino ---
mostrar_ruta_pasos(mejor_individuo, laberinto=laberinto,
                   inicio=inicio, salida=salida, grid=True)

#### Tasa de cruce media = 0.5

In [ ]:
# ============================================
#   PARÁMETROS DEL EXPERIMENTO (EDITABLES)
# ============================================
POP_SIZE = 300      # Tamaño de la población
NGEN     = 300       # Número de generaciones
CXPB     = 0.5      # Probabilidad de cruce
MUTPB    = 0.5      # Probabilidad de mutación
IND_SIZE = 1000       # Tamaño del individuo (cantidad de movimientos)
# ============================================

# Registrar individuo y población con el IND_SIZE actual
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_move, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
# Estadísticas
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("min", np.min)
stats.register("avg", np.mean)

# Población inicial
random.seed()  # Cambiar a un entero para reproducibilidad
pop = toolbox.population(n=POP_SIZE)

# Ejecutar algoritmo genético
resultados, logbook = algorithms.eaSimple(
    pop, toolbox,
    cxpb=CXPB,
    mutpb=MUTPB,
    ngen=NGEN,
    stats=stats,
    verbose=False
)

# Mejor individuo
mejor_individuo = tools.selBest(pop, 1)[0]
mejor_fitness = mejor_individuo.fitness.values[0]

# Posición final del mejor individuo
pos_final = inicio
for mov in mejor_individuo:
    pos_final = mover(pos_final, MOVIMIENTOS[mov])

# Reporte
print("=" * 50)
print("PARÁMETROS DEL EXPERIMENTO")
print("=" * 50)
print(f"POP_SIZE : {POP_SIZE}")
print(f"NGEN     : {NGEN}")
print(f"CXPB     : {CXPB}")
print(f"MUTPB    : {MUTPB}")
print(f"IND_SIZE : {IND_SIZE}")
print("=" * 50)
print("RESULTADOS")
print("=" * 50)
print(f"Mejor aptitud (distancia)  : {mejor_fitness:.4f}")
print(f"Posición final alcanzada   : {pos_final}")
print(f"Posición de salida (meta)  : {salida}")
print(f"Distancia final a la meta  : {mejor_fitness:.4f}")
print("=" * 50)

In [ ]:
 # --- Gráfica de evolución ---
generaciones = logbook.select("gen")
minimos      = logbook.select("min")
promedios    = logbook.select("avg")

plt.figure(figsize=(8, 5))
plt.plot(generaciones, minimos, label="Mejor aptitud")
plt.plot(generaciones, promedios, label="Aptitud promedio", linestyle="--")
plt.xlabel("Generaciones")
plt.ylabel("Aptitud (distancia a la salida)")
plt.title(f"Evolución | POP={POP_SIZE}, NGEN={NGEN}, CXPB={CXPB}, MUTPB={MUTPB}, IND={IND_SIZE}")
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización del mejor camino ---
mostrar_ruta_pasos(mejor_individuo, laberinto=laberinto,
                   inicio=inicio, salida=salida, grid=True)

#### Tasa de cruce alta = 0.8

In [ ]:
# ============================================
#   PARÁMETROS DEL EXPERIMENTO (EDITABLES)
# ============================================
POP_SIZE = 300      # Tamaño de la población
NGEN     = 300       # Número de generaciones
CXPB     = 0.8      # Probabilidad de cruce
MUTPB    = 0.5      # Probabilidad de mutación
IND_SIZE = 1000       # Tamaño del individuo (cantidad de movimientos)
# ============================================

# Registrar individuo y población con el IND_SIZE actual
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_move, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
# Estadísticas
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("min", np.min)
stats.register("avg", np.mean)

# Población inicial
random.seed()  # Cambiar a un entero para reproducibilidad
pop = toolbox.population(n=POP_SIZE)

# Ejecutar algoritmo genético
resultados, logbook = algorithms.eaSimple(
    pop, toolbox,
    cxpb=CXPB,
    mutpb=MUTPB,
    ngen=NGEN,
    stats=stats,
    verbose=False
)

# Mejor individuo
mejor_individuo = tools.selBest(pop, 1)[0]
mejor_fitness = mejor_individuo.fitness.values[0]

# Posición final del mejor individuo
pos_final = inicio
for mov in mejor_individuo:
    pos_final = mover(pos_final, MOVIMIENTOS[mov])

# Reporte
print("=" * 50)
print("PARÁMETROS DEL EXPERIMENTO")
print("=" * 50)
print(f"POP_SIZE : {POP_SIZE}")
print(f"NGEN     : {NGEN}")
print(f"CXPB     : {CXPB}")
print(f"MUTPB    : {MUTPB}")
print(f"IND_SIZE : {IND_SIZE}")
print("=" * 50)
print("RESULTADOS")
print("=" * 50)
print(f"Mejor aptitud (distancia)  : {mejor_fitness:.4f}")
print(f"Posición final alcanzada   : {pos_final}")
print(f"Posición de salida (meta)  : {salida}")
print(f"Distancia final a la meta  : {mejor_fitness:.4f}")
print("=" * 50)

In [ ]:
 # --- Gráfica de evolución ---
generaciones = logbook.select("gen")
minimos      = logbook.select("min")
promedios    = logbook.select("avg")

plt.figure(figsize=(8, 5))
plt.plot(generaciones, minimos, label="Mejor aptitud")
plt.plot(generaciones, promedios, label="Aptitud promedio", linestyle="--")
plt.xlabel("Generaciones")
plt.ylabel("Aptitud (distancia a la salida)")
plt.title(f"Evolución | POP={POP_SIZE}, NGEN={NGEN}, CXPB={CXPB}, MUTPB={MUTPB}, IND={IND_SIZE}")
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización del mejor camino ---
mostrar_ruta_pasos(mejor_individuo, laberinto=laberinto,
                   inicio=inicio, salida=salida, grid=True)

#### Análisis: Tasa de cruce

En este experimento, los tres valores probados (0.1, 0.5 y 0.8) dieron exactamente el mismo resultado: la ruta se detiene siempre en el mismo muro. Esto tiene sentido si se piensa en lo que hace el cruce de dos puntos: toma dos cromosomas y combina sus segmentos. El problema es que cuando todos los "buenos" individuos de la población ya comparten el mismo defecto —chocar contra la primera pared visible en dirección a la salida—, combinarlos solo produce más variantes del mismo error.

Dicho de otra forma, el cruce es bueno para combinar características que funcionan, pero no tiene manera de generar movimientos completamente nuevos que no estén ya en la población. Para este laberinto, donde superar un obstáculo requiere temporalmente alejarse de la meta (algo que la función de aptitud castiga), el cruce simplemente no es el operador indicado para escapar del mínimo local. La exploración necesaria solo puede venir de la mutación o de una función de aptitud más informada.

### Experimentos: Tasa de mutación

#### Tasa de mutación baja = 0.1

In [ ]:
# ============================================
#   PARÁMETROS DEL EXPERIMENTO (EDITABLES)
# ============================================
POP_SIZE = 300      # Tamaño de la población
NGEN     = 300       # Número de generaciones
CXPB     = 0.5      # Probabilidad de cruce
MUTPB    = 0.1      # Probabilidad de mutación
IND_SIZE = 1000       # Tamaño del individuo (cantidad de movimientos)
# ============================================

# Registrar individuo y población con el IND_SIZE actual
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_move, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
# Estadísticas
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("min", np.min)
stats.register("avg", np.mean)

# Población inicial
random.seed()  # Cambiar a un entero para reproducibilidad
pop = toolbox.population(n=POP_SIZE)

# Ejecutar algoritmo genético
resultados, logbook = algorithms.eaSimple(
    pop, toolbox,
    cxpb=CXPB,
    mutpb=MUTPB,
    ngen=NGEN,
    stats=stats,
    verbose=False
)

# Mejor individuo
mejor_individuo = tools.selBest(pop, 1)[0]
mejor_fitness = mejor_individuo.fitness.values[0]

# Posición final del mejor individuo
pos_final = inicio
for mov in mejor_individuo:
    pos_final = mover(pos_final, MOVIMIENTOS[mov])

# Reporte
print("=" * 50)
print("PARÁMETROS DEL EXPERIMENTO")
print("=" * 50)
print(f"POP_SIZE : {POP_SIZE}")
print(f"NGEN     : {NGEN}")
print(f"CXPB     : {CXPB}")
print(f"MUTPB    : {MUTPB}")
print(f"IND_SIZE : {IND_SIZE}")
print("=" * 50)
print("RESULTADOS")
print("=" * 50)
print(f"Mejor aptitud (distancia)  : {mejor_fitness:.4f}")
print(f"Posición final alcanzada   : {pos_final}")
print(f"Posición de salida (meta)  : {salida}")
print(f"Distancia final a la meta  : {mejor_fitness:.4f}")
print("=" * 50)

In [ ]:
 # --- Gráfica de evolución ---
generaciones = logbook.select("gen")
minimos      = logbook.select("min")
promedios    = logbook.select("avg")

plt.figure(figsize=(8, 5))
plt.plot(generaciones, minimos, label="Mejor aptitud")
plt.plot(generaciones, promedios, label="Aptitud promedio", linestyle="--")
plt.xlabel("Generaciones")
plt.ylabel("Aptitud (distancia a la salida)")
plt.title(f"Evolución | POP={POP_SIZE}, NGEN={NGEN}, CXPB={CXPB}, MUTPB={MUTPB}, IND={IND_SIZE}")
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización del mejor camino ---
mostrar_ruta_pasos(mejor_individuo, laberinto=laberinto,
                   inicio=inicio, salida=salida, grid=True)

#### Tasa de mutación media = 0.5

In [ ]:
# ============================================
#   PARÁMETROS DEL EXPERIMENTO (EDITABLES)
# ============================================
POP_SIZE = 300      # Tamaño de la población
NGEN     = 300       # Número de generaciones
CXPB     = 0.5      # Probabilidad de cruce
MUTPB    = 0.5      # Probabilidad de mutación
IND_SIZE = 1000       # Tamaño del individuo (cantidad de movimientos)
# ============================================

# Registrar individuo y población con el IND_SIZE actual
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_move, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
# Estadísticas
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("min", np.min)
stats.register("avg", np.mean)

# Población inicial
random.seed()  # Cambiar a un entero para reproducibilidad
pop = toolbox.population(n=POP_SIZE)

# Ejecutar algoritmo genético
resultados, logbook = algorithms.eaSimple(
    pop, toolbox,
    cxpb=CXPB,
    mutpb=MUTPB,
    ngen=NGEN,
    stats=stats,
    verbose=False
)

# Mejor individuo
mejor_individuo = tools.selBest(pop, 1)[0]
mejor_fitness = mejor_individuo.fitness.values[0]

# Posición final del mejor individuo
pos_final = inicio
for mov in mejor_individuo:
    pos_final = mover(pos_final, MOVIMIENTOS[mov])

# Reporte
print("=" * 50)
print("PARÁMETROS DEL EXPERIMENTO")
print("=" * 50)
print(f"POP_SIZE : {POP_SIZE}")
print(f"NGEN     : {NGEN}")
print(f"CXPB     : {CXPB}")
print(f"MUTPB    : {MUTPB}")
print(f"IND_SIZE : {IND_SIZE}")
print("=" * 50)
print("RESULTADOS")
print("=" * 50)
print(f"Mejor aptitud (distancia)  : {mejor_fitness:.4f}")
print(f"Posición final alcanzada   : {pos_final}")
print(f"Posición de salida (meta)  : {salida}")
print(f"Distancia final a la meta  : {mejor_fitness:.4f}")
print("=" * 50)

In [ ]:
 # --- Gráfica de evolución ---
generaciones = logbook.select("gen")
minimos      = logbook.select("min")
promedios    = logbook.select("avg")

plt.figure(figsize=(8, 5))
plt.plot(generaciones, minimos, label="Mejor aptitud")
plt.plot(generaciones, promedios, label="Aptitud promedio", linestyle="--")
plt.xlabel("Generaciones")
plt.ylabel("Aptitud (distancia a la salida)")
plt.title(f"Evolución | POP={POP_SIZE}, NGEN={NGEN}, CXPB={CXPB}, MUTPB={MUTPB}, IND={IND_SIZE}")
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización del mejor camino ---
mostrar_ruta_pasos(mejor_individuo, laberinto=laberinto,
                   inicio=inicio, salida=salida, grid=True)

#### Tasa de mutación alta = 0.8

In [ ]:
# ============================================
#   PARÁMETROS DEL EXPERIMENTO (EDITABLES)
# ============================================
POP_SIZE = 300      # Tamaño de la población
NGEN     = 300       # Número de generaciones
CXPB     = 0.5      # Probabilidad de cruce
MUTPB    = 0.8      # Probabilidad de mutación
IND_SIZE = 1000       # Tamaño del individuo (cantidad de movimientos)
# ============================================

# Registrar individuo y población con el IND_SIZE actual
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_move, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
# Estadísticas
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("min", np.min)
stats.register("avg", np.mean)

# Población inicial
random.seed()  # Cambiar a un entero para reproducibilidad
pop = toolbox.population(n=POP_SIZE)

# Ejecutar algoritmo genético
resultados, logbook = algorithms.eaSimple(
    pop, toolbox,
    cxpb=CXPB,
    mutpb=MUTPB,
    ngen=NGEN,
    stats=stats,
    verbose=False
)

# Mejor individuo
mejor_individuo = tools.selBest(pop, 1)[0]
mejor_fitness = mejor_individuo.fitness.values[0]

# Posición final del mejor individuo
pos_final = inicio
for mov in mejor_individuo:
    pos_final = mover(pos_final, MOVIMIENTOS[mov])

# Reporte
print("=" * 50)
print("PARÁMETROS DEL EXPERIMENTO")
print("=" * 50)
print(f"POP_SIZE : {POP_SIZE}")
print(f"NGEN     : {NGEN}")
print(f"CXPB     : {CXPB}")
print(f"MUTPB    : {MUTPB}")
print(f"IND_SIZE : {IND_SIZE}")
print("=" * 50)
print("RESULTADOS")
print("=" * 50)
print(f"Mejor aptitud (distancia)  : {mejor_fitness:.4f}")
print(f"Posición final alcanzada   : {pos_final}")
print(f"Posición de salida (meta)  : {salida}")
print(f"Distancia final a la meta  : {mejor_fitness:.4f}")
print("=" * 50)

In [ ]:
 # --- Gráfica de evolución ---
generaciones = logbook.select("gen")
minimos      = logbook.select("min")
promedios    = logbook.select("avg")

plt.figure(figsize=(8, 5))
plt.plot(generaciones, minimos, label="Mejor aptitud")
plt.plot(generaciones, promedios, label="Aptitud promedio", linestyle="--")
plt.xlabel("Generaciones")
plt.ylabel("Aptitud (distancia a la salida)")
plt.title(f"Evolución | POP={POP_SIZE}, NGEN={NGEN}, CXPB={CXPB}, MUTPB={MUTPB}, IND={IND_SIZE}")
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización del mejor camino ---
mostrar_ruta_pasos(mejor_individuo, laberinto=laberinto,
                   inicio=inicio, salida=salida, grid=True)

#### Análisis: Tasa de mutación

Quizás este es el resultado más frustrante del taller: subir la mutación de 0.1 a 0.8 no movió el marcador ni un milímetro. Las tres configuraciones quedaron atascadas en exactamente el mismo punto.

La razón es un poco paradójica. La mutación `mutShuffleIndexes` reorganiza los movimientos dentro del cromosoma, lo cual introduce variedad, pero no cambia el conjunto de movimientos disponibles. Si los individuos ya tienen pocas instrucciones útiles para navegar el laberinto y muchas que los hacen chocar o quedarse quietos, mezclarlas de distintas formas no mejora el resultado. Peor aún, cualquier mutante que accidentalmente se aleje de la meta para bordear un muro obtiene una distancia peor que su padre, así que la selección por torneo lo descarta de inmediato. La mutación sola no puede luchar contra una función de aptitud que penaliza exactamente los movimientos necesarios para avanzar.

### Experimentos: Número de pasos (tamaño del cromosoma)

#### Número bajo de pasos = 100

In [ ]:
# ============================================
#   PARÁMETROS DEL EXPERIMENTO (EDITABLES)
# ============================================
POP_SIZE = 300      # Tamaño de la población
NGEN     = 300       # Número de generaciones
CXPB     = 0.5      # Probabilidad de cruce
MUTPB    = 0.5      # Probabilidad de mutación
IND_SIZE = 100       # Tamaño del individuo (cantidad de movimientos)
# ============================================

# Registrar individuo y población con el IND_SIZE actual
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_move, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
# Estadísticas
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("min", np.min)
stats.register("avg", np.mean)

# Población inicial
random.seed()  # Cambiar a un entero para reproducibilidad
pop = toolbox.population(n=POP_SIZE)

# Ejecutar algoritmo genético
resultados, logbook = algorithms.eaSimple(
    pop, toolbox,
    cxpb=CXPB,
    mutpb=MUTPB,
    ngen=NGEN,
    stats=stats,
    verbose=False
)

# Mejor individuo
mejor_individuo = tools.selBest(pop, 1)[0]
mejor_fitness = mejor_individuo.fitness.values[0]

# Posición final del mejor individuo
pos_final = inicio
for mov in mejor_individuo:
    pos_final = mover(pos_final, MOVIMIENTOS[mov])

# Reporte
print("=" * 50)
print("PARÁMETROS DEL EXPERIMENTO")
print("=" * 50)
print(f"POP_SIZE : {POP_SIZE}")
print(f"NGEN     : {NGEN}")
print(f"CXPB     : {CXPB}")
print(f"MUTPB    : {MUTPB}")
print(f"IND_SIZE : {IND_SIZE}")
print("=" * 50)
print("RESULTADOS")
print("=" * 50)
print(f"Mejor aptitud (distancia)  : {mejor_fitness:.4f}")
print(f"Posición final alcanzada   : {pos_final}")
print(f"Posición de salida (meta)  : {salida}")
print(f"Distancia final a la meta  : {mejor_fitness:.4f}")
print("=" * 50)

In [ ]:
 # --- Gráfica de evolución ---
generaciones = logbook.select("gen")
minimos      = logbook.select("min")
promedios    = logbook.select("avg")

plt.figure(figsize=(8, 5))
plt.plot(generaciones, minimos, label="Mejor aptitud")
plt.plot(generaciones, promedios, label="Aptitud promedio", linestyle="--")
plt.xlabel("Generaciones")
plt.ylabel("Aptitud (distancia a la salida)")
plt.title(f"Evolución | POP={POP_SIZE}, NGEN={NGEN}, CXPB={CXPB}, MUTPB={MUTPB}, IND={IND_SIZE}")
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización del mejor camino ---
mostrar_ruta_pasos(mejor_individuo, laberinto=laberinto,
                   inicio=inicio, salida=salida, grid=True)

#### Número de pasos medio = 500

In [ ]:
# ============================================
#   PARÁMETROS DEL EXPERIMENTO (EDITABLES)
# ============================================
POP_SIZE = 300      # Tamaño de la población
NGEN     = 300       # Número de generaciones
CXPB     = 0.5      # Probabilidad de cruce
MUTPB    = 0.5      # Probabilidad de mutación
IND_SIZE = 500       # Tamaño del individuo (cantidad de movimientos)
# ============================================

# Registrar individuo y población con el IND_SIZE actual
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_move, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
# Estadísticas
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("min", np.min)
stats.register("avg", np.mean)

# Población inicial
random.seed()  # Cambiar a un entero para reproducibilidad
pop = toolbox.population(n=POP_SIZE)

# Ejecutar algoritmo genético
resultados, logbook = algorithms.eaSimple(
    pop, toolbox,
    cxpb=CXPB,
    mutpb=MUTPB,
    ngen=NGEN,
    stats=stats,
    verbose=False
)

# Mejor individuo
mejor_individuo = tools.selBest(pop, 1)[0]
mejor_fitness = mejor_individuo.fitness.values[0]

# Posición final del mejor individuo
pos_final = inicio
for mov in mejor_individuo:
    pos_final = mover(pos_final, MOVIMIENTOS[mov])

# Reporte
print("=" * 50)
print("PARÁMETROS DEL EXPERIMENTO")
print("=" * 50)
print(f"POP_SIZE : {POP_SIZE}")
print(f"NGEN     : {NGEN}")
print(f"CXPB     : {CXPB}")
print(f"MUTPB    : {MUTPB}")
print(f"IND_SIZE : {IND_SIZE}")
print("=" * 50)
print("RESULTADOS")
print("=" * 50)
print(f"Mejor aptitud (distancia)  : {mejor_fitness:.4f}")
print(f"Posición final alcanzada   : {pos_final}")
print(f"Posición de salida (meta)  : {salida}")
print(f"Distancia final a la meta  : {mejor_fitness:.4f}")
print("=" * 50)

In [ ]:
 # --- Gráfica de evolución ---
generaciones = logbook.select("gen")
minimos      = logbook.select("min")
promedios    = logbook.select("avg")

plt.figure(figsize=(8, 5))
plt.plot(generaciones, minimos, label="Mejor aptitud")
plt.plot(generaciones, promedios, label="Aptitud promedio", linestyle="--")
plt.xlabel("Generaciones")
plt.ylabel("Aptitud (distancia a la salida)")
plt.title(f"Evolución | POP={POP_SIZE}, NGEN={NGEN}, CXPB={CXPB}, MUTPB={MUTPB}, IND={IND_SIZE}")
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización del mejor camino ---
mostrar_ruta_pasos(mejor_individuo, laberinto=laberinto,
                   inicio=inicio, salida=salida, grid=True)

#### Número de pasos grande = 1300

In [ ]:
# ============================================
#   PARÁMETROS DEL EXPERIMENTO (EDITABLES)
# ============================================
POP_SIZE = 300      # Tamaño de la población
NGEN     = 300       # Número de generaciones
CXPB     = 0.5      # Probabilidad de cruce
MUTPB    = 0.5      # Probabilidad de mutación
IND_SIZE = 1300       # Tamaño del individuo (cantidad de movimientos)
# ============================================

# Registrar individuo y población con el IND_SIZE actual
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_move, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
# Estadísticas
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("min", np.min)
stats.register("avg", np.mean)

# Población inicial
random.seed()  # Cambiar a un entero para reproducibilidad
pop = toolbox.population(n=POP_SIZE)

# Ejecutar algoritmo genético
resultados, logbook = algorithms.eaSimple(
    pop, toolbox,
    cxpb=CXPB,
    mutpb=MUTPB,
    ngen=NGEN,
    stats=stats,
    verbose=False
)

# Mejor individuo
mejor_individuo = tools.selBest(pop, 1)[0]
mejor_fitness = mejor_individuo.fitness.values[0]

# Posición final del mejor individuo
pos_final = inicio
for mov in mejor_individuo:
    pos_final = mover(pos_final, MOVIMIENTOS[mov])

# Reporte
print("=" * 50)
print("PARÁMETROS DEL EXPERIMENTO")
print("=" * 50)
print(f"POP_SIZE : {POP_SIZE}")
print(f"NGEN     : {NGEN}")
print(f"CXPB     : {CXPB}")
print(f"MUTPB    : {MUTPB}")
print(f"IND_SIZE : {IND_SIZE}")
print("=" * 50)
print("RESULTADOS")
print("=" * 50)
print(f"Mejor aptitud (distancia)  : {mejor_fitness:.4f}")
print(f"Posición final alcanzada   : {pos_final}")
print(f"Posición de salida (meta)  : {salida}")
print(f"Distancia final a la meta  : {mejor_fitness:.4f}")
print("=" * 50)

In [ ]:
 # --- Gráfica de evolución ---
generaciones = logbook.select("gen")
minimos      = logbook.select("min")
promedios    = logbook.select("avg")

plt.figure(figsize=(8, 5))
plt.plot(generaciones, minimos, label="Mejor aptitud")
plt.plot(generaciones, promedios, label="Aptitud promedio", linestyle="--")
plt.xlabel("Generaciones")
plt.ylabel("Aptitud (distancia a la salida)")
plt.title(f"Evolución | POP={POP_SIZE}, NGEN={NGEN}, CXPB={CXPB}, MUTPB={MUTPB}, IND={IND_SIZE}")
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización del mejor camino ---
mostrar_ruta_pasos(mejor_individuo, laberinto=laberinto,
                   inicio=inicio, salida=salida, grid=True)

#### Análisis: Número de pasos

Este resultó ser el parámetro que más impacto tuvo en todos los experimentos. Con 100 pasos el algoritmo ni siquiera llega al primer muro (distancia 19), lo cual tiene una explicación sencilla: el laberinto es de 20x20 con paredes que obligan a dar rodeos largos, y 100 movimientos simplemente no alcanzan para explorar nada útil.

Cuando se aumenta a 500 y luego a 1300, la distancia va bajando porque los individuos tienen físicamente más oportunidades de moverse. La clave acá es que con muchos pasos disponibles, un individuo puede avanzar hacia la pared, rebotar, explorar lateralmente, y eventualmente pasar por el hueco casi sin que la evolución tenga que "planear" esa secuencia: la longitud del cromosoma compensa la falta de dirección inteligente.

En cierta forma, este resultado muestra el lado débil del enfoque: en lugar de aprender una ruta eficiente, el algoritmo termina dependiendo de que el cromosoma sea suficientemente largo como para que la solución aparezca por azar en algún segmento. Funciona, pero no es elegante ni escalable a laberintos más grandes.

## Conclusiones Generales

Después de correr todos los experimentos, lo que más llama la atención no son los resultados individuales de cada parámetro, sino el patrón que se repite en casi todos: el algoritmo converge rápido hacia un mínimo local y después no hay manera de sacarlo de ahí simplemente ajustando hiperparámetros.

**Sobre la función de aptitud.** El problema central no está en los operadores evolutivos, sino en cómo se evalúa cada individuo. Usar la distancia euclidiana directa a la salida funciona bien en espacios abiertos, pero en un laberinto con paredes es una señal engañosa: para rodear cualquier muro hay que alejarse momentáneamente de la meta, y eso baja el puntaje. El algoritmo aprende a no hacer eso, quedándose atrapado. Una función de aptitud que recompense exploración o que use distancia de camino (no en línea recta) cambiaría el comportamiento de fondo sin tocar ningún otro parámetro.

**Sobre la población y generaciones.** Más individuos dan más diversidad inicial, pero una vez que toda la población converge hacia la pared, el tamaño ya no importa. Las generaciones adicionales tampoco rompen la convergencia prematura; solo alargan el tiempo de cómputo. Lo que se observa en las curvas es que la mejora ocurre al principio y luego la línea se aplana, a veces desde muy temprano.

**Sobre cruce y mutación.** El cruce de dos puntos no introduce información nueva, solo mezcla lo que ya hay. Si los padres de élite comparten el mismo defecto, sus hijos también lo tendrán. La mutación por intercambio de índices (`mutShuffleIndexes`) tampoco ayuda mucho porque reordena movimientos existentes sin cambiar su tipo. Una mutación que reemplace genes por valores completamente nuevos (`mutUniformInt`) probablemente daría más exploración real.

**El parámetro que sí importó.** El tamaño del cromosoma (número de pasos) fue lo único que consistentemente cambió los resultados, pero por una razón que no es muy satisfactoria: con más pasos, la solución aparece por fuerza bruta probabilística, no porque el algoritmo haya aprendido algo mejor. Es un parche, no una solución.

En general, el ejercicio deja claro que un algoritmo genético puede ser muy sensible al diseño del problema (representación y función de aptitud) y relativamente poco sensible a los hiperparámetros clásicos cuando ese diseño tiene fallas estructurales. Antes de ajustar tasas de cruce o tamaños de población, vale la pena preguntarse si la función de aptitud está guiando al algoritmo en la dirección correcta.

## Modelo Final con los aprendizajes

Según lo aprendido en los experimentos, vamos a proponer un modelo el cual con lo que se evidenció, se espera que pueda llegar a la meta y lograr el último salto para salir de ese último mínimo local.

In [ ]:
# ============================================
#   PARÁMETROS DEL EXPERIMENTO (EDITABLES)
# ============================================
POP_SIZE = 500      # Tamaño de la población
NGEN     = 600       # Número de generaciones
CXPB     = 0.7      # Probabilidad de cruce
MUTPB    = 0.5      # Probabilidad de mutación
IND_SIZE = 2200       # Tamaño del individuo (cantidad de movimientos)
# ============================================

# Registrar individuo y población con el IND_SIZE actual
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_move, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
# Estadísticas
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("min", np.min)
stats.register("avg", np.mean)

# Población inicial
random.seed()  # Cambiar a un entero para reproducibilidad
pop = toolbox.population(n=POP_SIZE)

# Ejecutar algoritmo genético
resultados, logbook = algorithms.eaSimple(
    pop, toolbox,
    cxpb=CXPB,
    mutpb=MUTPB,
    ngen=NGEN,
    stats=stats,
    verbose=False
)

# Mejor individuo
mejor_individuo = tools.selBest(pop, 1)[0]
mejor_fitness = mejor_individuo.fitness.values[0]

# Posición final del mejor individuo
pos_final = inicio
for mov in mejor_individuo:
    pos_final = mover(pos_final, MOVIMIENTOS[mov])

# Reporte
print("=" * 50)
print("PARÁMETROS DEL EXPERIMENTO")
print("=" * 50)
print(f"POP_SIZE : {POP_SIZE}")
print(f"NGEN     : {NGEN}")
print(f"CXPB     : {CXPB}")
print(f"MUTPB    : {MUTPB}")
print(f"IND_SIZE : {IND_SIZE}")
print("=" * 50)
print("RESULTADOS")
print("=" * 50)
print(f"Mejor aptitud (distancia)  : {mejor_fitness:.4f}")
print(f"Posición final alcanzada   : {pos_final}")
print(f"Posición de salida (meta)  : {salida}")
print(f"Distancia final a la meta  : {mejor_fitness:.4f}")
print("=" * 50)

In [ ]:
 # --- Gráfica de evolución ---
generaciones = logbook.select("gen")
minimos      = logbook.select("min")
promedios    = logbook.select("avg")

plt.figure(figsize=(8, 5))
plt.plot(generaciones, minimos, label="Mejor aptitud")
plt.plot(generaciones, promedios, label="Aptitud promedio", linestyle="--")
plt.xlabel("Generaciones")
plt.ylabel("Aptitud (distancia a la salida)")
plt.title(f"Evolución | POP={POP_SIZE}, NGEN={NGEN}, CXPB={CXPB}, MUTPB={MUTPB}, IND={IND_SIZE}")
plt.legend()
plt.grid(True)
plt.show()

# --- Visualización del mejor camino ---
mostrar_ruta_pasos(mejor_individuo, laberinto=laberinto,
                   inicio=inicio, salida=salida, grid=True)